In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import gymnasium_driving

import src.environments.helpers.random

In [ ]:
env = gymnasium_driving.CarEnvironment(
    model="bicycle",
    road_network=gymnasium_driving.components.roads.RoadNetwork(
        roads=[gymnasium_driving.components.roads.Road(
            segments=[
                gymnasium_driving.components.roads.StraightSegment(
                    start=(10.0, 50.0),
                    heading=0.0,
                    length=100,
                )
            ],
            width=8.0,
        )],
    ),
    # road_network=gymnasium_driving.components.roads.RoadNetwork(
    #     roads=[
    #         gymnasium_driving.components.roads.create_rectangular_track(
    #             center=(50, 50),
    #             length=75,
    #             height=75,
    #             turn_radius=8.0,
    #             width=8.0,
    #         )
    #     ],
    # ),
    render_mode="rgb_array",
    proportion=(0.90, 0.00),
    noise=(0.0, 0.0),
    obstacles=[],
)
env = src.environments.helpers.random.RandomWaypointObstacles(env)

In [ ]:
env.reset()
helpers.preview(env)

In [ ]:
print("/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-13/19-38-59")
print("/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-14/15-40-08")
print("/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-14/14-08-35")

In [ ]:
# NOTE: able to avoid obstacles
# /Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-13/19-38-59
# /Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-14/15-40-08

# NOTE: able to follow a path
# /Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-14/14-08-35

# HYDRA_FULL_ERROR=1 python train.py \
#   environment@train=straight \
#   environment@eval=straight \
#   train.discrete=true \
#   eval.discrete=true
#   controller=dqn \
#   reward=path_progress_obstacles \
#   total_timesteps=1000000 \

In [ ]:
import sys

if "../" not in sys.path:
    sys.path.append("../")

In [ ]:
print("[sys.path]:", sys.path)

In [ ]:
import os
import time
import tqdm

import src
import src.helpers as helpers
import src.environments.cristal as cristal

import src.controllers.dqn as dqn
import src.controllers.ppo as ppo
import src.controllers.clothoids as clothoids
import src.controllers.purepursuit as purepursuit

import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

In [ ]:
# path = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-09/13-59-36/"

# path = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-08/21-12-59" # learned wit a static obstacle
# path = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-08/20-54-02" # learned with random obstacles

In [ ]:

output_directory = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-09/14-50-04"
output_directory = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/multirun/2026-02-09/15-26-19"
output_directory = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-09/16-30-57"

# NOTE: very good path follower, no obstacle avoidance
output_directory = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-09/19-05-24"

output_directory = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-09/19-45-59"

output_directory = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-09/21-56-50"

output_directory = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-12/15-25-48/"

output_directory = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-12/16-33-29"

output_directory = helpers.get_last_run_directory(
    base_directory="/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs",
    script="train"
)

output_directory = "/Users/nadir/Documents/research-project/code/experimentations/gymnasium/outputs/2026-02-15/17-04-24"

print("[output_directory]:", output_directory)

best_model_path = os.path.join(output_directory, "best_model.zip")
print("[best_model_exists]:", os.path.exists(best_model_path), best_model_path)

configuration = helpers.load_configuration(
    output_directory=output_directory,
    expected_script="train",
)

controller, train_environment, eval_environment = helpers.instantiate_configuration(
    configuration=configuration,
    output_directory=output_directory,
    load_best_model=True
)

environment = train_environment

environment.unwrapped.render_mode = "rgb_array"

In [ ]:
configuration.controller._target_

In [ ]:
configuration.reward

In [ ]:
environment.reset()

helpers.preview(environment)

In [ ]:
observation, info = environment.reset()

total_reward = 0

for i in range(500):
    action, _states = controller.get_action(observation=observation)
    
    observation, reward, terminated, truncated, info = environment.step(action)
    
    total_reward += reward
    
    environment.unwrapped.overlay_manager.clear()
    controller.draw_debug(environment=environment, observation=observation, path=environment.unwrapped.path)
    
    helpers.preview(environment)
    
    if terminated or truncated:
        break

environment.close()

print(total_reward)